In [0]:
def log_dq_results(catalog, schema, dq_results):
    """
    Batch logs data quality check results to etl_data_quality_log.
    """
    dq_log_table = f"{catalog}.{schema}.etl_data_quality_log"

    if not dq_results:
        logger.info("No DQ results to log")
        return
    
    log_schema = StructType([
        StructField("run_id", StringType(), False),
        StructField("table_name", StringType(), False),
        StructField("rule_name", StringType(), False),
        StructField("check_type", StringType(), True),
        StructField("sql_expression", StringType(), True),
        StructField("severity", StringType(), True),
        StructField("result", StringType(), True),
        StructField("fail_count", LongType(), True),
        StructField("message", StringType(), True),
        StructField("log_datetime", TimestampType(), True),
    ])

    try:
        rows = []
        for r in dq_results:
            rows.append({
                "run_id": r["run_id"],
                "table_name": r["table_name"],
                "rule_name": r["rule_name"],
                "check_type": r["check_type"],
                "sql_expression": r["sql_expression"],
                "severity": r["severity"],
                "result": r["result"],
                "fail_count": r["fail_count"],
                "message": r.get("message"),
                "log_datetime": datetime.now(),
            })

        df = spark.createDataFrame(rows, schema=log_schema)
        df.write.mode("append").format("delta").saveAsTable(dq_log_table)
        logger.info(f"Logged {len(rows)} DQ results")
    except Exception as e:
        logger.error(f"Failed to log DQ results: {e}")
        raise